In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import os, random
import numpy as np
import open3d as o3d
from pathlib import Path

# OCC 라이브러리
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.BRepAdaptor import BRepAdaptor_Surface, BRepAdaptor_Curve
from OCC.Core.TopExp import TopExp_Explorer
from OCC.Core.TopAbs import TopAbs_FACE, TopAbs_EDGE, TopAbs_VERTEX, TopAbs_REVERSED
from OCC.Core.BRep import BRep_Tool
from OCC.Core.GeomAbs import (
    GeomAbs_Plane, GeomAbs_Cylinder, GeomAbs_Cone, 
    GeomAbs_Sphere, GeomAbs_Torus, GeomAbs_BezierSurface, 
    GeomAbs_BSplineSurface, GeomAbs_SurfaceOfRevolution, 
    GeomAbs_SurfaceOfExtrusion, GeomAbs_OffsetSurface, 
    GeomAbs_OtherSurface, GeomAbs_Line, GeomAbs_Circle,
    GeomAbs_Ellipse, GeomAbs_Hyperbola, GeomAbs_Parabola,
    GeomAbs_BezierCurve, GeomAbs_BSplineCurve
)

# OCC 관련 모듈 (STEP 파싱용)
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from OCC.Core.TopLoc import TopLoc_Location
from OCC.Core.GCPnts import GCPnts_QuasiUniformDeflection

# 곡면 타입별 색상 지정 (RGB 0~1)
COLOR_MAP = {
    GeomAbs_Plane: [0.8, 0.8, 0.8],               # 평면 (밝은 회색)
    GeomAbs_Cylinder: [0.2, 0.6, 1.0],            # 원기둥 (파란색)
    GeomAbs_Cone: [1.0, 0.8, 0.0],                # 원뿔 (노란색)
    GeomAbs_Sphere: [0.2, 0.8, 0.2],              # 구 (초록색)
    GeomAbs_Torus: [1.0, 0.5, 0.0],               # 토러스/도넛형 (주황색)
    GeomAbs_BSplineSurface: [1.0, 0.2, 0.2],      # B-스플라인 (빨간색)
    GeomAbs_SurfaceOfRevolution: [0.6, 0.2, 0.8], # 회전체 (보라색)
    GeomAbs_SurfaceOfExtrusion: [0.4, 0.8, 0.8],  # 돌출체 (청록색)
    GeomAbs_BezierSurface: [1.0, 0.4, 0.7],       # 베지어 곡면 (분홍색)
    GeomAbs_OffsetSurface: [0.5, 0.5, 0.0],       # 오프셋 곡면 (올리브색)
    GeomAbs_OtherSurface: [0.3, 0.3, 0.3]         # 기타/복합 곡면 (아주 짙은 회색)
}
# 예외처리 색상
DEFAULT_COLOR = [0.0, 0.0, 0.0] # 완전 검은색 (에러)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
class ABCMultiModalDataset(Dataset):
    def __init__(self, base_dir, pcd_num_points=2048, voxel_res=64, max_faces=100, max_edges=500):
        self.base_dir = Path(base_dir)
        self.pcd_num_points = pcd_num_points
        self.voxel_res = voxel_res
        self.max_faces = max_faces
        self.max_edges = max_edges
        self.device = o3d.core.Device("CPU:0")
        
        self.obj_files = []
        print("OBJ file scanning...")
        for chunk_folder in sorted(os.listdir(self.base_dir)):
            chunk_path = self.base_dir / chunk_folder
            if chunk_path.is_dir():
                for model_id in os.listdir(chunk_path):
                    model_path = chunk_path / model_id
                    objs = list(model_path.glob("*.obj"))
                    if objs:
                        self.obj_files.append(objs[0])
        print(f"{len(self.obj_files)} OBJ files loaded.")

    def __len__(self):
        return len(self.obj_files)

    def _extract_brep_target(self, step_path):
        """STEP 파일에서 모든 GeomAbs 타입을 고려하여 파라미터를 안전하게 추출"""
        reader = STEPControl_Reader()
        if reader.ReadFile(str(step_path)) != 1:
            return torch.zeros((self.max_faces, 7)), torch.zeros((self.max_edges, 3)), torch.zeros(1)
            
        reader.TransferRoots()
        shape = reader.OneShape()
        
        # 1. Face Target 추출 (7개 컬럼으로 확장: Type, P1, P2, P3, Ox, Oy, Oz)
        faces_data = []
        exp = TopExp_Explorer(shape, TopAbs_FACE)
        while exp.More():
            face = exp.Current()
            surf = BRepAdaptor_Surface(face)
            stype = surf.GetType()
            # [TypeIndex, Param1, Param2, Param3, OriginX, OriginY, OriginZ]
            f_vec = [float(stype), 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
            
            try:
                if stype == GeomAbs_Plane:
                    loc = surf.Plane().Location()
                    f_vec[4:7] = [loc.X(), loc.Y(), loc.Z()]
                elif stype == GeomAbs_Cylinder:
                    cyl = surf.Cylinder()
                    f_vec[1] = cyl.Radius()
                    loc = cyl.Location()
                    f_vec[4:7] = [loc.X(), loc.Y(), loc.Z()]
                elif stype == GeomAbs_Cone:
                    cone = surf.Cone()
                    f_vec[1] = cone.RefRadius()
                    f_vec[2] = cone.SemiAngle()
                    loc = cone.Location()
                    f_vec[4:7] = [loc.X(), loc.Y(), loc.Z()]
                elif stype == GeomAbs_Sphere:
                    sph = surf.Sphere()
                    f_vec[1] = sph.Radius()
                    loc = sph.Location()
                    f_vec[4:7] = [loc.X(), loc.Y(), loc.Z()]
                elif stype == GeomAbs_Torus:
                    tor = surf.Torus()
                    f_vec[1] = tor.MajorRadius()
                    f_vec[2] = tor.MinorRadius()
                    loc = tor.Location()
                    f_vec[4:7] = [loc.X(), loc.Y(), loc.Z()]
                elif stype == GeomAbs_BezierSurface:
                    bez = surf.Bezier()
                    f_vec[1] = float(bez.NbUPoles())
                    f_vec[2] = float(bez.NbVPoles())
                elif stype == GeomAbs_BSplineSurface:
                    bspl = surf.BSpline()
                    f_vec[1] = float(bspl.DegreeU())
                    f_vec[2] = float(bspl.DegreeV())
                    f_vec[3] = float(bspl.NbUPoles() * bspl.NbVPoles()) # 제어점 총합
                elif stype in [GeomAbs_SurfaceOfRevolution, GeomAbs_SurfaceOfExtrusion]:
                    # 생성 곡선의 타입 정보를 P1에 기록
                    f_vec[1] = float(surf.BasisCurve().GetType())
            except Exception:
                pass # 추출 실패 시 기본값 0 유지
                
            faces_data.append(f_vec)
            exp.Next()
            if len(faces_data) >= self.max_faces: break
            
        # 2. Edge Target 추출 (3개 컬럼: Type, Radius/Param, Length)
        edges_data = []
        exp = TopExp_Explorer(shape, TopAbs_EDGE)
        while exp.More():
            edge = exp.Current()
            curve = BRepAdaptor_Curve(edge)
            ctype = curve.GetType()
            e_vec = [float(ctype), 0.0, 0.0]
            
            try:
                # 곡선 길이 계산 (간이)
                e_vec[2] = curve.LastParameter() - curve.FirstParameter()
                
                if ctype == GeomAbs_Circle:
                    e_vec[1] = curve.Circle().Radius()
                elif ctype == GeomAbs_Ellipse:
                    e_vec[1] = curve.Ellipse().MajorRadius()
                elif ctype == GeomAbs_BSplineCurve:
                    e_vec[1] = float(curve.BSpline().Degree())
            except Exception:
                pass
                
            edges_data.append(e_vec)
            exp.Next()
            if len(edges_data) >= self.max_edges: break

        # 3. Vertex Count
        v_count = 0
        exp = TopExp_Explorer(shape, TopAbs_VERTEX)
        while exp.More():
            v_count += 1
            exp.Next()

        # 텐서 크기 고정 및 패딩
        faces_tensor = np.zeros((self.max_faces, 7))
        if faces_data: faces_tensor[:len(faces_data)] = faces_data
        edges_tensor = np.zeros((self.max_edges, 3))
        if edges_data: edges_tensor[:len(edges_data)] = edges_data

        return torch.tensor(faces_tensor).float(), torch.tensor(edges_tensor).float(), torch.tensor([v_count]).float()

    def _simulate_stereo_depth(self, mesh, max_mesh_dist):
        """물체 크기에 비례한 동적 카메라 거리 설정 (작은 물체 해상도 문제 해결)"""
        dist_min = max_mesh_dist * 2.5
        dist_max = max_mesh_dist * 4.0
        
        phi = np.random.uniform(0, 2*np.pi)
        theta = np.arccos(np.random.uniform(-1, 1))
        radius = np.random.uniform(dist_min, dist_max)
        
        cam_pos = np.array([
            radius * np.sin(theta) * np.cos(phi),
            radius * np.sin(theta) * np.sin(phi),
            radius * np.cos(theta)
        ])
        
        scene = o3d.t.geometry.RaycastingScene()
        scene.add_triangles(o3d.t.geometry.TriangleMesh.from_legacy(mesh, device=self.device))
        
        # 해상도를 고정하여 물체가 작을수록 더 세밀하게 캡처되도록 유도
        rays = scene.create_rays_pinhole(fov_deg=69, center=[0,0,0], eye=cam_pos, up=[0,1,0], width_px=320, height_px=240)
        ans = scene.cast_rays(rays)
        t_hit = ans['t_hit'].numpy()
        hit_mask, rays_arr = np.isfinite(t_hit), rays.numpy()
        pts = rays_arr[hit_mask][:, :3] + t_hit[hit_mask][:, np.newaxis] * rays_arr[hit_mask][:, 3:]
        
        z_depths = np.dot(pts - cam_pos, -cam_pos/np.linalg.norm(cam_pos))
        noise = np.random.normal(0, 1, size=pts.shape) * ((z_depths**2 * 0.08) / (800 * 50))[:, np.newaxis]
        return pts + noise

    def __getitem__(self, idx):
        obj_path = self.obj_files[idx]
        step_path = obj_path.with_suffix('.step')
        
        mesh = o3d.io.read_triangle_mesh(str(obj_path))
        mesh.translate(-mesh.get_center())
        max_mesh_dist = np.max(np.linalg.norm(np.asarray(mesh.vertices), axis=1))
        
        raw_sim_points = self._simulate_stereo_depth(mesh, max_mesh_dist)
        pcd_pts = raw_sim_points / (max_mesh_dist + 1e-8)
        
        if len(pcd_pts) >= self.pcd_num_points:
            idx_sample = np.random.choice(len(pcd_pts), self.pcd_num_points, replace=False)
        else:
            idx_sample = np.random.choice(len(pcd_pts), self.pcd_num_points, replace=True)
        pcd_pts = pcd_pts[idx_sample]
        
        voxel_pts = (pcd_pts * 0.5) + 0.5 
        voxel_indices = np.clip(((voxel_pts * (self.voxel_res - 1)).astype(int)), 0, self.voxel_res - 1)
        voxel_matrix = np.zeros((self.voxel_res, self.voxel_res, self.voxel_res), dtype=np.float32)
        voxel_matrix[voxel_indices[:, 0], voxel_indices[:, 1], voxel_indices[:, 2]] = 1.0

        face_tgt, edge_tgt, v_count_tgt = self._extract_brep_target(step_path)
        
        return {
            "pcd": torch.from_numpy(pcd_pts).float(),
            "voxel": torch.from_numpy(voxel_matrix).float().unsqueeze(0),
            "target_faces": face_tgt,
            "target_edges": edge_tgt,
            "target_vcount": v_count_tgt,
            "model_id": obj_path.parent.name,
            "obj_path": str(obj_path),
            "step_path": str(step_path),
            "max_mesh_dist": max_mesh_dist 
        }

In [4]:
# --- 사용 예시 ---
dataset = ABCMultiModalDataset(base_dir="./abc_dataset_filtered-1", pcd_num_points=2048, voxel_res=64)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=4)

OBJ file scanning...
74234 OBJ files loaded.


In [5]:
# --- 시각화: OBJ, PCD, Voxel ---
sample = dataset[random.randint(0, len(dataset)-1)]
# sample = dataset[]

# 1. 원본 Mesh 준비 및 데이터셋과 동일한 방식으로 정규화
orig_mesh = o3d.io.read_triangle_mesh(sample['obj_path'])
orig_mesh.translate(-orig_mesh.get_center()) # 원점 이동
# 데이터셋에서 사용한 동일한 max_mesh_dist로 스케일 조정
orig_mesh.scale(1.0 / sample['max_mesh_dist'], center=(0, 0, 0))
orig_mesh.paint_uniform_color([0.7, 0.7, 0.7]) # 회색

# 2. 시뮬레이션된 PCD (이미 정규화되어 있음)
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(sample['pcd'].numpy())
pcd.paint_uniform_color([1, 0, 0]) # 빨간색

# 3. Voxel 시각화 데이터 (검증용)
vox_matrix = sample['voxel'].squeeze().numpy()
indices = np.argwhere(vox_matrix == 1.0)
# 인덱스 공간 [0, 63] -> 정규화 공간 [-1, 1]로 복원
v_pts = (indices / (dataset.voxel_res - 1) - 0.5) * 2.0

voxel_pcd = o3d.geometry.PointCloud()
voxel_pcd.points = o3d.utility.Vector3dVector(v_pts)
voxel_pcd.paint_uniform_color([0, 1, 0]) # 초록색

# 시각화 실행
# Mesh와 PCD는 겹쳐서(Overlay), Voxel은 옆으로 이동시켜서 비교
print(f"Model ID: {sample['model_id']}")
print("Center Overlay: Grey Mesh + Red PCD")
print("Right Offset: Green Voxel Grid")

o3d.visualization.draw_geometries([orig_mesh, pcd, voxel_pcd.translate([2.5, 0, 0])],
                                  window_name="Mesh, PCD, Voxel",
                                  width=1200, height=800)

Model ID: 00091868
Center Overlay: Grey Mesh + Red PCD
Right Offset: Green Voxel Grid


In [1]:
# 1. 데이터 샘플 준비
sample = dataset[random.randint(0, len(dataset)-1)]
# sample = dataset[0]

obj_path = Path(sample['obj_path'])
step_path = Path(sample['step_path'])
max_dist = sample['max_mesh_dist']

print(f"Visualizing Model ID: {sample['model_id']}")

# --- A. 원본 OBJ Mesh (회색 반투명용) ---
orig_mesh = o3d.io.read_triangle_mesh(str(obj_path))
orig_mesh.translate(-orig_mesh.get_center())
orig_mesh.scale(1.0 / max_dist, center=(0, 0, 0))
orig_mesh.paint_uniform_color([0.7, 0.7, 0.7])

# --- B. 시뮬레이션 PCD (빨간색) ---
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(sample['pcd'].numpy())
pcd.paint_uniform_color([1, 0, 0])

# --- C. Voxel 데이터 (초록색, 오른쪽 오프셋) ---
vox_matrix = sample['voxel'].squeeze().numpy()
indices = np.argwhere(vox_matrix == 1.0)
v_pts = (indices / (dataset.voxel_res - 1) - 0.5) * 2.0
voxel_pcd = o3d.geometry.PointCloud()
voxel_pcd.points = o3d.utility.Vector3dVector(v_pts)
voxel_pcd.paint_uniform_color([0, 1, 0])
voxel_pcd.translate([2, 0, 0]) # 오른쪽으로 이동

# --- D. STEP 기반 B-Rep 시각화 데이터 생성 ---
reader = STEPControl_Reader()
reader.ReadFile(str(step_path))
reader.TransferRoots()
shape = reader.OneShape()
BRepMesh_IncrementalMesh(shape, 0.05)

# D-1. Face Primitive 메쉬 생성
step_combined_mesh = o3d.geometry.TriangleMesh()
explorer_face = TopExp_Explorer(shape, TopAbs_FACE)
while explorer_face.More():
    face = explorer_face.Current()
    loc = TopLoc_Location()
    triangulation = BRep_Tool.Triangulation(face, loc)
    if triangulation:
        adaptor = BRepAdaptor_Surface(face)
        surf_type = adaptor.GetType()
        face_color = COLOR_MAP.get(surf_type, DEFAULT_COLOR)
        
        verts = []
        for i in range(1, triangulation.NbNodes() + 1):
            p = triangulation.Node(i).Transformed(loc.Transformation())
            verts.append([p.X(), p.Y(), p.Z()])
            
        tris = []
        is_reversed = (face.Orientation() == TopAbs_REVERSED)
        for i in range(1, triangulation.NbTriangles() + 1):
            t = triangulation.Triangle(i)
            n1, n2, n3 = t.Get()
            if is_reversed: tris.append([n1-1, n3-1, n2-1])
            else: tris.append([n1-1, n2-1, n3-1])
            
        f_mesh = o3d.geometry.TriangleMesh()
        f_mesh.vertices = o3d.utility.Vector3dVector(np.array(verts))
        f_mesh.triangles = o3d.utility.Vector3iVector(np.array(tris))
        f_mesh.paint_uniform_color(face_color)
        step_combined_mesh += f_mesh
    explorer_face.Next()

# D-2. Edge 와이어프레임 생성
edge_points, edge_lines = [], []
point_idx = 0
explorer_edge = TopExp_Explorer(shape, TopAbs_EDGE)
while explorer_edge.More():
    edge = explorer_edge.Current()
    adaptor = BRepAdaptor_Curve(edge)
    discretizer = GCPnts_QuasiUniformDeflection(adaptor, 0.01)
    if discretizer.IsDone():
        pts = [[discretizer.Value(i).X(), discretizer.Value(i).Y(), discretizer.Value(i).Z()] 
               for i in range(1, discretizer.NbPoints() + 1)]
        if len(pts) > 1:
            edge_points.extend(pts)
            for i in range(len(pts) - 1):
                edge_lines.append([point_idx + i, point_idx + i + 1])
            point_idx += len(pts)
    explorer_edge.Next()

wireframe = o3d.geometry.LineSet()
wireframe.points = o3d.utility.Vector3dVector(np.array(edge_points))
wireframe.lines = o3d.utility.Vector2iVector(np.array(edge_lines))
wireframe.paint_uniform_color([0, 0, 0]) # 검정색 와이어프레임

# D-3. STEP 데이터 정규화 (OBJ/PCD와 일치시키기)
# STEP 데이터는 정규화 전이므로, OBJ 메쉬가 사용했던 원점과 스케일을 그대로 적용
# (dataset 코드에서 mesh.get_center()를 기준으로 했으므로 동일하게 수행)
step_center = step_combined_mesh.get_center()
step_combined_mesh.translate(-step_center)
step_combined_mesh.scale(1.0 / max_dist, center=(0, 0, 0))
wireframe.translate(-step_center)
wireframe.scale(1.0 / max_dist, center=(0, 0, 0))

# 왼쪽으로 이동시켜서 원본 OBJ/PCD와 분리해서 보기 (선택 사항)
# 겹쳐보고 싶다면 translate를 주석 처리하세요.
step_combined_mesh.translate([-2, 0, 0])
wireframe.translate([-2, 0, 0])

# --- 2. 최종 시각화 실행 ---
print("Rendered:")
print(" - LEFT  : STEP B-Rep Class (Primitive Color + Black Wireframe)")
print(" - MIDDLE: OBJ Mesh(Grey) + Sim. PCD(Red) overlay")
print(" - RIGHT : Sim. Voxel Grid(Green)")

o3d.visualization.draw_geometries(
    [orig_mesh, pcd, voxel_pcd, step_combined_mesh, wireframe],
    window_name=f"Comprehensive CAD Data Check: {sample['model_id']}",
    width=1400, height=900,
    mesh_show_back_face=True
)

NameError: name 'dataset' is not defined